In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import time

## Simular una data de matrices de Atención

Parametros

In [2]:
n_batch = 4
n_embed = 10
context_length = 8
vocab_size = 40

In [4]:
# Datos de entrada
data = torch.randint(vocab_size, (n_batch, context_length))
print(data)

tensor([[ 2, 11, 17, 32,  1, 36, 10,  8],
        [39,  4, 19, 11, 25,  1,  8,  1],
        [15, 22, 11, 21,  0, 27, 26,  5],
        [26, 19, 10, 32, 21,  3, 28, 14]])


In [5]:
embeddings = nn.Embedding(vocab_size, n_embed)
x = embeddings(data)
print(x.shape)
print(x)

torch.Size([4, 8, 10])
tensor([[[ 3.6692e-01,  1.6613e-01,  5.3340e-01, -4.2437e-01, -4.3459e-01,
           3.9925e-01,  1.1136e+00, -8.2850e-01,  1.1339e+00, -9.7280e-01],
         [-8.5007e-01,  6.1662e-01, -7.3050e-01, -1.4198e+00,  1.5941e+00,
           4.0423e-01, -1.4204e+00,  9.9889e-01, -9.4444e-01,  1.9196e-01],
         [ 4.5201e-01,  5.4953e-01,  7.7975e-01,  2.2576e-01, -8.5711e-01,
           5.3229e-02, -3.5794e-01, -1.5359e+00,  7.3779e-01,  6.8140e-01],
         [ 8.8110e-01,  3.7571e-01,  3.3671e-01, -1.7683e+00, -9.4713e-01,
           8.8714e-01,  4.5031e-01, -9.6052e-01,  3.3057e-01, -3.7252e-01],
         [-4.3614e-01,  2.2424e+00, -1.5305e-01, -3.0657e-01,  9.5549e-01,
           1.1377e+00, -9.8238e-01,  6.0244e-01,  7.0966e-01,  8.8591e-01],
         [-9.2921e-01, -9.5552e-01, -1.2327e+00,  8.5620e-02, -7.2789e-01,
          -7.8826e-01, -9.3255e-01, -5.5882e-01,  9.0208e-01, -5.4584e-01],
         [ 2.2867e-01,  1.0658e-01,  1.7450e-01, -1.3004e+00,  1.0160e+

In [6]:
# Creando los valores de las Matrices QKV

In [7]:
key = nn.Linear(n_embed, n_embed, bias=False)
query = nn.Linear(n_embed, n_embed, bias=False)
value = nn.Linear(n_embed, n_embed, bias=False)

In [8]:
# como estamos tratando de un Single Head Atenttion el tamaño de la matriz cuadrada es de emdeddings por embeddings (10 x 10)

Procesando los datos

In [9]:
k = query(x)
q = key(x)
v = value(x)

In [11]:
# Imprimir dimensiones de los datos
print(f'     Matriz de datos: {data.shape}')
print(f'Matriz de embeddings: {embeddings.weight.shape}')
print(f' Embeddings de tokens: {k.shape}')
print(f'Embeddings de posición: {q.shape}')
print(f'  Embeddings de valores: {v.shape}')

     Matriz de datos: torch.Size([4, 8])
Matriz de embeddings: torch.Size([40, 10])
 Embeddings de tokens: torch.Size([4, 8, 10])
Embeddings de posición: torch.Size([4, 8, 10])
  Embeddings de valores: torch.Size([4, 8, 10])


La matriz de datos se divide en : batch size x sequence Lenght (donde seran enteros correspondientes a los tokens)

Los Embeddings son 40 x 10 (vocab size x embeddings) esta es la matriz de embeddigs no no los vectores de nuestros datos.

Los Tokens Embeddigs: es el batch size x el context lenght por la dimension de los embeddings

In [12]:
print(f'      tamaño de Q: {query.weight.shape}')
print(f'      tamaño de K: {key.weight.shape}')
print(f'      tamaño de V: {value.weight.shape}')

      tamaño de Q: torch.Size([10, 10])
      tamaño de K: torch.Size([10, 10])
      tamaño de V: torch.Size([10, 10])


In [13]:
print(f'      tamaño de Q: {q.shape}')
print(f'      tamaño de K: {k.shape}')
print(f'      tamaño de V: {v.shape}')

      tamaño de Q: torch.Size([4, 8, 10])
      tamaño de K: torch.Size([4, 8, 10])
      tamaño de V: torch.Size([4, 8, 10])


Aca corresponde a batch size por el sequence lenght x la dimensionalidad de los embeddings

# Implementar self-attention manual

In [14]:
### Implementación manual

# "Similitud del coseno" entre query y keys (nota: sería similitud del coseno si se escalara por |q||k|)
qk = q @ k.transpose(
    -2, -1
)  # transponer dimensiones que no son de batch; la única parte acá es asegurarte de que la primera dimensión es el batch y no se modifica

# Escalar QK por su varianza
qk_scaled = (
    qk * n_embed**-0.5
)  # a la potencia de -0.5 (que equivale a dividir entre la raíz cuadrada de n_embed: qk / torch.sqrt(torch.tensor(n_embed)))

# Aplicar máscara para tokens futuros
pastmask = torch.tril(torch.ones(n_batch, context_length, context_length))
qk_scaled[pastmask == 0] = -torch.inf  # equivalente a sumar una matriz de ceros/-inf

# Aplicar softmax
qk_softmax = F.softmax(qk_scaled, dim=-1)

# Y mecanismo final de atención
actsManual = (
    qk_softmax @ v
)  # después de tener la máscara y aplicar softmax, multiplicamos por v
print(
    f"Forma de las activaciones (manual): {actsManual.shape}"
)  # [batch, context, n_embed]

Forma de las activaciones (manual): torch.Size([4, 8, 10])


## Implementarlo con Pytorch

In [15]:
# Implementación con PyTorch
actsTorch = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f'Forma de las activaciones (PyTorch): {actsTorch.shape}')

Forma de las activaciones (PyTorch): torch.Size([4, 8, 10])


In [16]:
# Comparar
print(actsManual[0, :, :])
print('')
print(actsTorch[0, :, :])
print('')
print(actsManual[0, :, :] - actsTorch[0, :, :])

print(f'\n\n¿Son _exactamente_ iguales? {torch.equal(actsManual, actsTorch)}')
print(
    f'¿Son "iguales" (dentro de la tolerancia)?'
    f' {torch.allclose(actsManual, actsTorch)}'
)

tensor([[ 0.3038,  0.6826,  0.5421,  0.3029,  0.4135,  0.2742, -0.1878, -0.0669,
         -0.7122,  0.1176],
        [ 0.1685, -0.5085, -0.4547, -0.1086, -0.2940, -0.0673,  0.3992,  0.2775,
         -0.3943, -0.3386],
        [-0.0611,  0.1544,  0.4330,  0.1457,  0.1649,  0.0345, -0.2230, -0.0138,
         -0.5088,  0.1414],
        [-0.0665,  0.3647,  0.4483,  0.3396,  0.2032,  0.0410, -0.1584,  0.0760,
         -0.5936,  0.0733],
        [-0.0433,  0.1576,  0.2027,  0.1454, -0.0136,  0.0122, -0.0579,  0.1491,
         -0.5858,  0.0783],
        [-0.0506,  0.0627,  0.2550,  0.2242, -0.0771, -0.0045, -0.1024,  0.0595,
         -0.5136,  0.1024],
        [ 0.0481, -0.1562,  0.1907,  0.1131, -0.0050, -0.0239,  0.0668, -0.0076,
         -0.4870,  0.0052],
        [ 0.0239, -0.1886,  0.2795,  0.2259, -0.0290, -0.0461,  0.0912,  0.0159,
         -0.3584, -0.0544]], grad_fn=<SelectBackward0>)

tensor([[ 0.3038,  0.6826,  0.5421,  0.3029,  0.4135,  0.2742, -0.1878, -0.0669,
         -0.7122, 

Aunque no son una coincidencia perfecta cuando colocamos la funcion torch.equal? no son exactamente iguales debido a algunos errores de calculo de aplicacion y redondeo pero van a ser muy parecidos dicho todo esto la opcion de pytoch tendria mas estabilidad estadistica que nuestros calculos manuales asi que es mas fiable porque esta literalmente optimizado

## Cual de los dos es mas rapido?

In [17]:
numReps = 50_000

# Versión manual
start_time = time.time()
for _ in range(numReps):
  qk = q @ k.transpose(-2, -1) * (n_embed**-0.5)
  pastmask = torch.tril(torch.ones(n_batch, context_length, context_length))
  qk[pastmask == 0] = -torch.inf
  qk = F.softmax(qk, dim=-1)
  activations = qk @ v
print(f'---    Manual: {time.time()-start_time:.3f} s')

# Versión optimizada (PyTorch)
start_time = time.time()
for _ in range(numReps):
  activations = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f'--- Optimizada: {time.time()-start_time:.3f} s')

---    Manual: 7.547 s
--- Optimizada: 5.826 s


## con GPU

In [19]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [20]:
device

device(type='cuda', index=0)

In [21]:
# Parámetros
n_batch = 64
n_embed = 1000
context_length = 1024  # Reducido context_length para que quepa en memoria
vocab_size = 50257

# Crear matrices
data = torch.randint(
    vocab_size,
    (n_batch, context_length),
    dtype=torch.long,
    device=device,
)
embedding = nn.Embedding(vocab_size, n_embed, device=device)
key = nn.Linear(n_embed, n_embed, bias=False, device=device)
query = nn.Linear(n_embed, n_embed, bias=False, device=device)
value = nn.Linear(n_embed, n_embed, bias=False, device=device)

x = embedding(data)
k = key(x)
q = query(x)
v = value(x)

In [22]:
x

tensor([[[-2.6030e-01, -2.2725e+00,  1.1535e-01,  ..., -2.3501e-01,
           5.3907e-01, -6.2543e-01],
         [-1.0968e+00,  4.4771e-01,  4.1195e-02,  ..., -3.3937e-01,
           1.5146e+00, -4.0059e-01],
         [ 7.6539e-01,  1.2313e+00,  4.5566e-01,  ..., -2.1133e+00,
          -4.1846e-01, -1.5593e+00],
         ...,
         [ 1.3528e+00, -1.5606e+00,  1.6957e-01,  ..., -3.7480e-01,
           8.6306e-01,  5.1042e-01],
         [ 6.1454e-01,  1.1442e+00,  4.1670e-01,  ...,  1.4781e-01,
          -2.6020e-01,  1.8787e+00],
         [-1.0136e+00, -4.0324e-01,  1.0603e+00,  ...,  2.4048e-01,
           2.3754e-01,  5.7822e-01]],

        [[ 8.3788e-01, -2.8701e-01, -4.8488e-01,  ...,  2.4317e+00,
          -5.2335e-01,  1.1037e+00],
         [-8.4134e-01, -7.5580e-02, -9.3799e-01,  ..., -2.5991e-01,
          -1.2661e+00,  5.5679e-01],
         [-1.2513e-01,  3.7189e-01, -6.5402e-01,  ..., -1.4803e+00,
          -7.6000e-01, -2.4606e-01],
         ...,
         [-5.8099e-01, -7

##· Para el test

In [23]:
numReps = 200

torch.cuda.synchronize()  # Sincroniza GPU y CPU. Bueno para pruebas de tiempo, malo para el rendimiento general
start_time = time.time()
for _ in range(numReps):
  qk = q @ k.transpose(-2, -1) * (n_embed**-0.5)
  pastmask = torch.tril(
      torch.ones(n_batch, context_length, context_length, device=device)
  )
  qk[pastmask == 0] = -torch.inf
  qk_softmax = F.softmax(qk, dim=-1)
  activationsM = qk_softmax @ v
print(f'--- Manual:  {time.time()-start_time:.3f} s')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsT = F.scaled_dot_product_attention(
      q, k, v, is_causal=True, scale=n_embed**-0.5
  )
print(f'--- PyTorch: {time.time()-start_time:.3f} s')

--- Manual:  6.330 s
--- PyTorch: 12.350 s


hay un hallazgo interesante Pytorch es en realidad mas lento que la implementación Manual, lo que pasa con la GPU es que hay un tiempo de inicialización hay un cierto coste por lo que siempre es bueno cuando se esta ejecutando un clock time test (se puede ejecutar dos veces para ver que esta pasando)

In [24]:
# OPTIMIZACION ADICIONAL

basicamente lo que estamos haciendo es compilar la funcion de dot scale product atention usando el Just in time de Pytorch compiler

y renombrando la funcion SDPA_compiled y setear los valores a flotantes

In [25]:
# Algunas optimizaciones adicionales
import torch._dynamo

SDPA_compiled = torch.compile(F.scaled_dot_product_attention)
torch.set_float32_matmul_precision('high')

In [ ]:
# Otra tecnica de Optimizacion del algoritmo llamada FYI, FlashAttention: https://github.com/Dao-AILab/fl

In [26]:
numReps = 200

torch.cuda.synchronize()  # Sincroniza GPU y CPU. Bueno para pruebas de tiempo, malo para el rendimiento general
start_time = time.time()
for _ in range(numReps):
  qk = q @ k.transpose(-2, -1) * (n_embed**-0.5)
  pastmask = torch.tril(
      torch.ones(n_batch, context_length, context_length, device=device)
  )
  qk[pastmask == 0] = -torch.inf
  qk = F.softmax(qk, dim=-1)
  activationsM = qk @ v
print(f'--- Manual:    {time.time()-start_time:.3f} s')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsP = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f'--- PyTorch:   {time.time()-start_time:.3f} s')


torch.cuda.synchronize()
start_time = time.time()
for _ in range(numReps):
  activationsO = SDPA_compiled(q, k, v, is_causal=True)
print(f'--- Compilado: {time.time()-start_time:.3f} s')

--- Manual:    7.173 s
--- PyTorch:   14.140 s


W0820 20:38:33.228000 1447 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


--- Compilado: 6.961 s


In [ ]:
# baja un poco mas el tiemp de calculo que viene de cambiar la precision de ma la matriz con los valores Float